[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/06-nlp-and-transformers/notebooks/03-hugging-face-pipelines.ipynb)

# 03 - Hugging Face pipelines, tokenizers, models and the Hub

**Goal:** use the Hugging Face `transformers` library at three levels: the one-line `pipeline`, tokenizer + model by hand, and reading what the model returns. We do **sentiment analysis** (is a text positive or negative?) and **text generation** with tiny models that run on a CPU.

Downloads about 550 MB the first time (two models). No API keys.

## Setup
Install anything missing, and fix the random seed so generated text is the same every run.

In [1]:
import importlib.util, subprocess, sys
# Install a package only if it is missing (Colab already has most of these).
for pkg, mod in [("transformers", "transformers"), ("sentence-transformers", "sentence_transformers"),
                 ("tiktoken", "tiktoken"), ("torch", "torch")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("Packages ready")
import torch
torch.manual_seed(0)

Packages ready


## 1. Level 1: `pipeline`

`pipeline(task, model=...)` bundles tokenizer + model + post-processing. We name the model explicitly; without a name the library picks a default and prints a warning. The model card for this one is on the Hub: https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english

In [2]:
from transformers import pipeline

clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
reviews = [
    "I love this course, the examples are so clear!",
    "The battery died after one hour. Terrible.",
    "It was okay, nothing special.",
    "Great, it broke on the first day.",
]
for r, res in zip(reviews, clf(reviews)):
    print(f"{res['label']:<8} {res['score']:.3f}  {r}")

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3812.74it/s]

POSITIVE 1.000  I love this course, the examples are so clear!
NEGATIVE 1.000  The battery died after one hour. Terrible.
NEGATIVE 0.982  It was okay, nothing special.
POSITIVE 1.000  Great, it broke on the first day.


**Look closely:** "It was okay" gets a confident label even though it is neutral — this model only knows two labels. "Great, it broke..." is sarcasm and is probably labelled POSITIVE. A high score means "confident", not "correct".

## 2. Level 2: tokenizer + model by hand

Same result, but each step is visible:
1. the tokenizer turns text into IDs (and an **attention mask** — 1 for real tokens, 0 for padding),
2. the model returns **logits** (raw scores),
3. softmax turns logits into probabilities.

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

inputs = tok(reviews, padding=True, return_tensors="pt")
print("input_ids shape:", tuple(inputs["input_ids"].shape), "(reviews, tokens incl. padding)")
print("first review tokens:", tok.convert_ids_to_tokens(inputs["input_ids"][0]))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7035.95it/s]

input_ids shape: (4, 13) (reviews, tokens incl. padding)
first review tokens: ['[CLS]', 'i', 'love', 'this', 'course', ',', 'the', 'examples', 'are', 'so', 'clear', '!', '[SEP]']


Now run the model. `torch.no_grad()` tells PyTorch we are not training, which saves memory and time.

In [4]:
with torch.no_grad():
    logits = model(**inputs).logits
probs = torch.softmax(logits, dim=-1)
print("labels:", model.config.id2label)
for r, l, p in zip(reviews, logits, probs):
    print(f"logits={l.numpy().round(2)}  probs={p.numpy().round(3)}  {r[:40]}")

labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
logits=[-4.31  4.66]  probs=[0. 1.]  I love this course, the examples are so 
logits=[ 4.5  -3.69]  probs=[1. 0.]  The battery died after one hour. Terribl
logits=[ 2.13 -1.88]  probs=[0.982 0.018]  It was okay, nothing special.
logits=[-3.81  4.09]  probs=[0. 1.]  Great, it broke on the first day.


## 3. What is inside the model?

A model is a stack of layers holding **parameters** (learned numbers). Let's count them and look at the config, which is downloaded from the Hub with the weights.

In [5]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.1f} million")
cfg = model.config
print("layers:", cfg.n_layers, " heads:", cfg.n_heads, " hidden size:", cfg.dim, " vocab:", cfg.vocab_size)

Parameters: 67.0 million
layers: 6  heads: 12  hidden size: 768  vocab: 30522


## 4. Text generation with a tiny model

`HuggingFaceTB/SmolLM2-135M-Instruct` is a **decoder-only** model (it writes text left to right) with 135 million parameters. It is tiny compared with chat models, so expect mistakes. `max_new_tokens` limits how much it writes; `do_sample=False` means "always pick the most likely next token" (called **greedy decoding**).

In [6]:
# dtype=torch.float32: the weights are stored in bfloat16, which is very slow on most CPUs
gen = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.float32)
out = gen("The three most useful Python libraries for data are", max_new_tokens=30, do_sample=False)
print(out[0]["generated_text"])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:  34%|███▍      | 93/272 [00:00<00:00, 923.70it/s]

Loading weights:  68%|██████▊   | 186/272 [00:00<00:00, 777.24it/s]

Loading weights:  98%|█████████▊| 266/272 [00:00<00:00, 786.23it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 803.77it/s]

The three most useful Python libraries for data are Pandas, NumPy, and Matplotlib. Pandas is a library for data manipulation and analysis, NumPy is a library for numerical computations, and Matplotlib is a library


## 5. Chat format

**Instruct** models were trained on conversations with roles: `system` (rules for the assistant), `user` (the person), `assistant` (the model). `apply_chat_template` wraps messages in the special markers the model expects. Let's print the wrapped text first so the markers are visible.

In [7]:
from transformers import AutoModelForCausalLM

chat_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
ctok = AutoTokenizer.from_pretrained(chat_name)
cmodel = AutoModelForCausalLM.from_pretrained(chat_name, dtype=torch.float32)
messages = [
    {"role": "system", "content": "You are a friendly teacher. Answer in one short sentence."},
    {"role": "user", "content": "What is sentiment analysis?"},
]
print(ctok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 40/272 [00:00<00:00, 391.55it/s]

Loading weights:  38%|███▊      | 104/272 [00:00<00:00, 439.42it/s]

Loading weights:  54%|█████▍    | 148/272 [00:00<00:00, 436.17it/s]

Loading weights:  71%|███████   | 192/272 [00:00<00:00, 417.86it/s]

Loading weights:  88%|████████▊ | 239/272 [00:00<00:00, 433.15it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 444.48it/s]

<|im_start|>system
You are a friendly teacher. Answer in one short sentence.<|im_end|>
<|im_start|>user
What is sentiment analysis?<|im_end|>
<|im_start|>assistant



Now generate the answer and decode only the new tokens (the prompt is part of the output IDs, so we slice it off).

In [8]:
inputs = ctok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
with torch.no_grad():
    new_ids = cmodel.generate(**inputs, max_new_tokens=40, do_sample=False)
print(ctok.decode(new_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Sentiment analysis is the process of analyzing a text to determine its emotional tone, sentiment, or attitude. It helps in identifying whether a message is positive, negative, or neutral.


## 6. Other pipeline tasks (to explore later)

The same pattern works for many tasks. Each needs its own model, so these are listed, not run, to keep downloads small:

| task string | what it does | example small model |
|---|---|---|
| `"ner"` | find names, places, organisations | `dslim/bert-base-NER` |
| `"zero-shot-classification"` | classify into labels you invent | `facebook/bart-large-mnli` (large) |
| `"summarization"` | shorten text | `sshleifer/distilbart-cnn-12-6` |
| `"translation_en_to_fr"` | translate | `t5-small` |
| `"feature-extraction"` | get embeddings | `sentence-transformers/all-MiniLM-L6-v2` |

Search the Hub by task: https://huggingface.co/models

## 7. Batch sentiment on a small table
A realistic use: score a column of customer comments with pandas and count the labels.

In [9]:
import pandas as pd

df = pd.DataFrame({"comment": [
    "Delivery was fast and the food was hot.",
    "Order arrived cold and late.",
    "Customer support solved my issue in minutes.",
    "The app keeps crashing when I pay.",
    "Decent value for the price.",
]})
results = clf(df["comment"].tolist())
df["label"] = [r["label"] for r in results]
df["score"] = [round(r["score"], 3) for r in results]
print(df)
print(df["label"].value_counts())

                                        comment     label  score
0       Delivery was fast and the food was hot.  POSITIVE  1.000
1                  Order arrived cold and late.  NEGATIVE  0.999
2  Customer support solved my issue in minutes.  NEGATIVE  0.971
3            The app keeps crashing when I pay.  NEGATIVE  0.996
4                   Decent value for the price.  POSITIVE  1.000
label
NEGATIVE    3
POSITIVE    2
Name: count, dtype: int64


## Try it yourself
1. Look at the table above. Did any plainly positive comment get NEGATIVE (when we built this notebook, "Customer support solved my issue in minutes." did)? Why might a model trained on movie reviews struggle with support tickets?
2. Write five tricky reviews (sarcasm, negation like "not bad", mixed feelings). Which ones fool the model?
3. Change `do_sample=False` to `do_sample=True, temperature=1.2` in section 4 and run it a few times. What changes? (Module 07 explains temperature.)
4. Open the model card for SmolLM2-135M-Instruct on the Hub. What licence does it have? What languages does it support?